**Persiapan Data & Feature Engineering**


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Load Data
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# 2. Feature Engineering
# a. Ekstraksi Title dari Nama
df['Title'] = df['Name'].apply(lambda x: re.search(' ([A-Za-z]+)\.', x).group(1))
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace('Mlle', 'Miss')
df['Title'] = df['Title'].replace('Ms', 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')
df['Title'] = df['Title'].map(title_mapping)

# b. Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# c. Imputasi Missing Values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# d. Binning Usia
df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 20, 40, 120], labels=['Child', 'Teen', 'Adult', 'Elder'])

# e. Encoding Kategorikal
le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex']) # 0: Female, 1: Male
df['AgeBin'] = le.fit_transform(df['AgeBin'])
df['Embarked'] = le.fit_transform(df['Embarked'])

# Drop kolom yang tidak digunakan
features_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Age']
df = df.drop(columns=features_to_drop)

# 3. Train-Test Split & Scaling
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature Engineering Selesai! Ukuran data latih:", X_train_scaled.shape)

<>:16: SyntaxWarning: invalid escape sequence '\.'
<>:16: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_1829/3771165867.py:16: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].apply(lambda x: re.search(' ([A-Za-z]+)\.', x).group(1))


Feature Engineering Selesai! Ukuran data latih: (712, 9)


**Baseline Model Konvensional**

In [ ]:
# --- Model 1: Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
print("=== Random Forest Accuracy ===")
print(accuracy_score(y_test, rf_pred))

# --- Model 2: XGBoost ---
xgb_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)
print("\n=== XGBoost Accuracy ===")
print(accuracy_score(y_test, xgb_pred))

=== Random Forest Accuracy ===
0.8212290502793296

=== XGBoost Accuracy ===
0.8100558659217877


**Model Deep Learning (MLP)**

menggunakan Multi-Layer Perceptron (MLP).

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Membangun Arsitektur MLP
mlp_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3), # Regularisasi mematikan 30% neuron

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(1, activation='sigmoid') # Sigmoid untuk klasifikasi biner
])

# Compile Model (Adam optimizer & Binary Crossentropy loss)
mlp_model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

# Early Stopping sebagai rem darurat
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training Model
history = mlp_model.fit(X_train_scaled, y_train,
                        validation_split=0.2,
                        epochs=100,
                        batch_size=32,
                        callbacks=[early_stop],
                        verbose=0) # Ubah verbose=1 jika ingin melihat proses per-epoch

# Evaluasi MLP
mlp_loss, mlp_acc = mlp_model.evaluate(X_test_scaled, y_test, verbose=0)
print("\n=== Keras MLP Accuracy ===")
print(mlp_acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



=== Keras MLP Accuracy ===
0.8156424760818481
